<a href="https://colab.research.google.com/github/MarriRohan/Quantum-Computing/blob/main/Hybrid_Quantum_Optimization_System_Product_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# PRODUCT 3: HYBRID QUANTUM OPTIMIZATION SYSTEM (SINGLE CODE FINAL)
# ==============================================================================

# Install dependencies in Google Colab:
!pip install qiskit qiskit-aer matplotlib scipy ipywidgets -q

import numpy as np
import scipy.optimize as opt
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

import qiskit
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import ParameterVector
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError

# ------------------------------------------------------------------------------
# 1. PROBLEM BENCHMARKS (Graph Max-Cut Formulation)
# ------------------------------------------------------------------------------
PROBLEMS = {
    "2-Node Toy Graph (1 Edge)": {
        "num_qubits": 2,
        "edges": [(0, 1)],
        "max_cut": 1,
        "optimal_bitstrings": ["01", "10"],
        "min_H_C": -1.0,
        "description": "Minimal 2-qubit benchmark: vertices 0 and 1 connected by an edge."
    },
    "3-Node Triangle Graph (3 Edges)": {
        "num_qubits": 3,
        "edges": [(0, 1), (1, 2), (0, 2)],
        "max_cut": 2,
        "optimal_bitstrings": ["001", "010", "100", "011", "101", "110"],
        "min_H_C": -1.0,
        "description": "Frustrated graph: Complete K3 graph where maximum cut is 2."
    },
    "4-Node Ring Graph (4 Edges)": {
        "num_qubits": 4,
        "edges": [(0, 1), (1, 2), (2, 3), (3, 0)],
        "max_cut": 4,
        "optimal_bitstrings": ["0101", "1010"],
        "min_H_C": -4.0,
        "description": "Bipartite cyclic graph: Alternating vertices yield a maximum cut of 4."
    }
}

# ------------------------------------------------------------------------------
# 2. PARAMETERIZED QUANTUM CIRCUIT BUILDERS
# ------------------------------------------------------------------------------
def build_qaoa_circuit(num_qubits, edges, p=1):
    """Constructs a QAOA ansatz with p layers of cost and mixer unitaries."""
    gammas = ParameterVector('gamma', p)
    betas = ParameterVector('beta', p)
    qc = QuantumCircuit(num_qubits)

    # Initial state: Equal superposition |+>
    qc.h(range(num_qubits))

    for i in range(p):
        # Problem Unitary: exp(-i * gamma * Z_u * Z_v)
        for u, v in edges:
            qc.cx(u, v)
            qc.rz(2 * gammas[i], v)
            qc.cx(u, v)
        # Mixer Unitary: exp(-i * beta * X_u)
        for u in range(num_qubits):
            qc.rx(2 * betas[i], u)

    param_list = list(gammas) + list(betas)
    return qc, param_list

def build_vqe_ansatz(num_qubits, reps=1):
    """Constructs a hardware-efficient RealAmplitudes style variational ansatz."""
    total_params = num_qubits * (reps + 1)
    thetas = ParameterVector('theta', total_params)
    qc = QuantumCircuit(num_qubits)

    idx = 0
    for q in range(num_qubits):
        qc.ry(thetas[idx], q)
        idx += 1

    for r in range(reps):
        for q in range(num_qubits - 1):
            qc.cx(q, q + 1)
        if num_qubits > 2:
            qc.cx(num_qubits - 1, 0)
        for q in range(num_qubits):
            qc.ry(thetas[idx], q)
            idx += 1

    return qc, list(thetas)

# ------------------------------------------------------------------------------
# 3. NOISE MODEL & ERROR MITIGATION ROUTINES
# ------------------------------------------------------------------------------
def create_noise_model(depol_1q=0.005, depol_2q=0.03, p_readout=0.05):
    """Creates a realistic NISQ noise model."""
    noise_model = NoiseModel()
    err_1 = depolarizing_error(depol_1q, 1)
    noise_model.add_all_qubit_quantum_error(err_1, ['rx', 'ry', 'rz', 'h', 'x', 'sx'])
    err_2 = depolarizing_error(depol_2q, 2)
    noise_model.add_all_qubit_quantum_error(err_2, ['cx'])
    readout_err = ReadoutError([[1 - p_readout, p_readout], [p_readout, 1 - p_readout]])
    noise_model.add_all_qubit_readout_error(readout_err)
    return noise_model

def get_calibration_matrix(backend, num_qubits, shots=1024):
    """Generates the empirical assignment matrix A for readout error mitigation."""
    dim = 2 ** num_qubits
    cal_matrix = np.zeros((dim, dim))
    for i in range(dim):
        bit_str = format(i, f'0{num_qubits}b')
        qc = QuantumCircuit(num_qubits)
        for q, bit in enumerate(reversed(bit_str)):
            if bit == '1':
                qc.x(q)
        qc.measure_all()
        t_qc = transpile(qc, backend)
        counts = backend.run(t_qc, shots=shots).result().get_counts()
        for sampled_str, cnt in counts.items():
            j = int(sampled_str, 2)
            cal_matrix[j, i] = cnt / shots
    return cal_matrix

def mitigate_counts(raw_counts, cal_matrix, num_qubits, shots):
    """Mitigates measurement readout noise via matrix inversion."""
    dim = 2 ** num_qubits
    v = np.zeros(dim)
    for bitstring, cnt in raw_counts.items():
        v[int(bitstring, 2)] = cnt / shots
    try:
        inv_cal = np.linalg.inv(cal_matrix)
        v_mit = inv_cal @ v
    except np.linalg.LinAlgError:
        v_mit = v
    # Project to valid probability distribution
    v_mit = np.clip(v_mit, 0, None)
    total = np.sum(v_mit)
    if total > 0:
        v_mit /= total
    mit_counts = {}
    for idx, prob in enumerate(v_mit):
        if prob > 1e-4:
            b_str = format(idx, f'0{num_qubits}b')
            mit_counts[b_str] = int(round(prob * shots))
    return mit_counts

# ------------------------------------------------------------------------------
# 4. EXPECTATION VALUES & PERFORMANCE METRICS
# ------------------------------------------------------------------------------
def compute_expectation(counts, edges, num_qubits):
    """Computes expected cost Hamiltonian energy <H_C> = sum_{(u,v)} Z_u Z_v."""
    total_shots = sum(counts.values())
    exp_val = 0.0
    for bitstr, cnt in counts.items():
        val = 0.0
        for u, v in edges:
            spin_u = 1 if bitstr[-(u+1)] == '0' else -1
            spin_v = 1 if bitstr[-(v+1)] == '0' else -1
            val += spin_u * spin_v
        exp_val += val * (cnt / total_shots)
    return exp_val

def compute_cut_metrics(counts, edges, num_qubits, max_cut, optimal_bitstrings):
    """Computes cut expectation, approximation ratio, and success probability."""
    total_shots = sum(counts.values())
    avg_cuts = 0.0
    success_shots = 0
    for bitstr, cnt in counts.items():
        cuts = sum(1 for u, v in edges if bitstr[-(u+1)] != bitstr[-(v+1)])
        avg_cuts += cuts * (cnt / total_shots)
        if bitstr in optimal_bitstrings:
            success_shots += cnt
    approx_ratio = avg_cuts / max_cut if max_cut > 0 else 0.0
    success_prob = success_shots / total_shots
    return avg_cuts, approx_ratio, success_prob

# ------------------------------------------------------------------------------
# 5. INTERACTIVE GUI CONTROLS (IPYWIDGETS)
# ------------------------------------------------------------------------------
dropdown_problem = widgets.Dropdown(
    options=list(PROBLEMS.keys()),
    value="3-Node Triangle Graph (3 Edges)",
    description="Problem Graph:",
    style={'description_width': '140px'},
    layout=widgets.Layout(width='450px')
)

dropdown_algo = widgets.Dropdown(
    options=["QAOA (p=1)", "QAOA (p=2)", "VQE (RealAmplitudes)"],
    value="QAOA (p=1)",
    description="Algorithm:",
    style={'description_width': '140px'},
    layout=widgets.Layout(width='450px')
)

dropdown_backend = widgets.Dropdown(
    options=[
        "Comparative Benchmark (Ideal vs Noisy vs Mitigated)",
        "Ideal Simulator (Aer)",
        "Noisy NISQ Simulator",
        "Error Mitigated Simulator"
    ],
    value="Comparative Benchmark (Ideal vs Noisy vs Mitigated)",
    description="Execution Mode:",
    style={'description_width': '140px'},
    layout=widgets.Layout(width='450px')
)

slider_shots = widgets.IntSlider(
    value=2048,
    min=512,
    max=4096,
    step=512,
    description="Measurement Shots:",
    style={'description_width': '140px'},
    layout=widgets.Layout(width='450px')
)

button_run = widgets.Button(
    description="Run Hybrid Optimization",
    button_style='primary',
    icon='play',
    layout=widgets.Layout(width='280px', height='42px', margin='10px 0 0 140px')
)

output_box = widgets.Output()

# ------------------------------------------------------------------------------
# 6. MAIN HYBRID OPTIMIZATION EXECUTION ENGINE
# ------------------------------------------------------------------------------
def execute_hybrid_system(b):
    with output_box:
        clear_output()
        prob_name = dropdown_problem.value
        algo_choice = dropdown_algo.value
        exec_mode = dropdown_backend.value
        shots = slider_shots.value

        prob_data = PROBLEMS[prob_name]
        nq = prob_data["num_qubits"]
        edges = prob_data["edges"]
        max_c = prob_data["max_cut"]
        opt_bits = prob_data["optimal_bitstrings"]
        min_hc = prob_data["min_H_C"]

        print("=" * 75)
        print("HYBRID QUANTUM OPTIMIZATION SYSTEM: EXECUTION REPORT")
        print("=" * 75)
        print(f"Selected Problem  : {prob_name}")
        print(f"Graph Description : {prob_data['description']}")
        print(f"Vertices / Qubits : {nq} | Graph Edges: {edges}")
        print(f"Theoretical Cut   : {max_c} (Ground State Energy H_C: {min_hc})")
        print(f"Optimal Solutions : {opt_bits}")
        print(f"Algorithm Selected: {algo_choice} | Execution Mode: {exec_mode}")
        print(f"Measurement Shots : {shots}")
        print("=" * 75)

        # Build Parameterized Circuit
        if algo_choice == "QAOA (p=1)":
            qc, params = build_qaoa_circuit(nq, edges, p=1)
            init_angles = np.array([0.5, 0.5])
        elif algo_choice == "QAOA (p=2)":
            qc, params = build_qaoa_circuit(nq, edges, p=2)
            init_angles = np.array([0.5, 0.5, 0.5, 0.5])
        else: # VQE
            qc, params = build_vqe_ansatz(nq, reps=1)
            init_angles = np.full(len(params), 0.3)

        print("\n[1] PARAMETERIZED QUANTUM CIRCUIT DIAGRAM:")
        print(qc.draw(output='text'))

        # Hardware Transpilation Analysis
        ideal_sim = AerSimulator()
        basis_gates = ['cx', 'id', 'rz', 'sx', 'x']
        t_qc = transpile(qc, ideal_sim, basis_gates=basis_gates)
        print("\n[2] HARDWARE TRANSPILATION ANALYSIS:")
        print(f"  • Original Circuit Depth   : {qc.depth()}")
        print(f"  • Transpiled Circuit Depth : {t_qc.depth()} (Basis: {basis_gates})")
        print(f"  • Two-Qubit CNOT Gate Count: {t_qc.count_ops().get('cx', 0)}")
        print(f"  • Total Transpiled Gates   : {sum(t_qc.count_ops().values())}")

        # Classical Optimization Loop (using COBYLA)
        print("\n[3] EXECUTING CLASSICAL-QUANTUM OPTIMIZATION LOOP (COBYLA)...")
        history_energies = []

        def objective(theta_vals):
            bind_dict = {params[i]: theta_vals[i] for i in range(len(params))}
            bound = qc.assign_parameters(bind_dict)
            bound.measure_all()
            transpiled = transpile(bound, ideal_sim)
            res = ideal_sim.run(transpiled, shots=1024).result()
            e_val = compute_expectation(res.get_counts(), edges, nq)
            history_energies.append(e_val)
            return e_val

        opt_result = opt.minimize(objective, init_angles, method='COBYLA', options={'maxiter': 25})
        opt_params = opt_result.x

        print(f"  • Classical Optimization Converged in {opt_result.nfev} function evaluations.")
        print(f"  • Optimal Parameters Found: {np.round(opt_params, 4)}")
        print(f"  • Optimized Ideal Energy  : {opt_result.fun:.4f}")

        # Prepare Bound Evaluation Circuit
        final_qc = qc.assign_parameters({params[i]: opt_params[i] for i in range(len(params))})
        final_qc.measure_all()

        # Prepare Simulation Backends
        noise_model = create_noise_model(depol_1q=0.005, depol_2q=0.03, p_readout=0.05)
        noisy_sim = AerSimulator(noise_model=noise_model)

        # Ideal Execution
        t_id = transpile(final_qc, ideal_sim)
        cnt_ideal = ideal_sim.run(t_id, shots=shots).result().get_counts()
        e_ideal = compute_expectation(cnt_ideal, edges, nq)
        cuts_id, ar_id, succ_id = compute_cut_metrics(cnt_ideal, edges, nq, max_c, opt_bits)

        # Noisy Execution
        t_no = transpile(final_qc, noisy_sim)
        cnt_noisy = noisy_sim.run(t_no, shots=shots).result().get_counts()
        e_noisy = compute_expectation(cnt_noisy, edges, nq)
        cuts_no, ar_no, succ_no = compute_cut_metrics(cnt_noisy, edges, nq, max_c, opt_bits)

        # Error Mitigated Execution
        cal_mat = get_calibration_matrix(noisy_sim, nq, shots=shots)
        cnt_mit = mitigate_counts(cnt_noisy, cal_mat, nq, shots=shots)
        e_mit = compute_expectation(cnt_mit, edges, nq)
        cuts_mit, ar_mit, succ_mit = compute_cut_metrics(cnt_mit, edges, nq, max_c, opt_bits)

        print("\n[4] PERFORMANCE METRICS EVALUATION:")
        print("-" * 75)
        print(f"{'Execution Mode':<25} | {'Energy <H_C>':<12} | {'Approx Ratio':<12} | {'Success Prob':<12}")
        print("-" * 75)
        print(f"{'Theoretical Exact':<25} | {min_hc:<12.4f} | {1.0:<12.4f} | {1.0:<12.4f}")
        print(f"{'Ideal Simulator':<25} | {e_ideal:<12.4f} | {ar_id:<12.4f} | {succ_id:<12.4f}")
        print(f"{'Noisy Simulator':<25} | {e_noisy:<12.4f} | {ar_no:<12.4f} | {succ_no:<12.4f}")
        print(f"{'Error Mitigated':<25} | {e_mit:<12.4f} | {ar_mit:<12.4f} | {succ_mit:<12.4f}")
        print("-" * 75)

        # Graphical Reporting: Plots
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

        # Plot 1: Optimization Convergence Trajectory
        ax1.plot(range(1, len(history_energies) + 1), history_energies, 'o-', color='#1f77b4', lw=2, label='COBYLA Energy')
        ax1.axhline(y=min_hc, color='red', linestyle='--', label=f'Exact Min Energy ({min_hc})')
        ax1.set_xlabel('Classical Optimization Iteration', fontsize=11)
        ax1.set_ylabel('Cost Hamiltonian Energy <H_C>', fontsize=11)
        ax1.set_title(f'Hybrid Convergence: {algo_choice}', fontsize=12, fontweight='bold')
        ax1.grid(True, alpha=0.3)
        ax1.legend()

        # Plot 2: Bitstring Measurement Distribution
        all_bitstrings = sorted(list(set(list(cnt_ideal.keys()) + list(cnt_noisy.keys()) + list(cnt_mit.keys()))))
        x_indices = np.arange(len(all_bitstrings))
        width = 0.27

        ideal_probs = [cnt_ideal.get(b, 0) / shots for b in all_bitstrings]
        noisy_probs = [cnt_noisy.get(b, 0) / shots for b in all_bitstrings]
        mit_probs = [cnt_mit.get(b, 0) / shots for b in all_bitstrings]

        ax2.bar(x_indices - width, ideal_probs, width, label='Ideal', color='#2ca02c', alpha=0.85)
        ax2.bar(x_indices, noisy_probs, width, label='Noisy NISQ', color='#d62728', alpha=0.85)
        ax2.bar(x_indices + width, mit_probs, width, label='Mitigated', color='#1f77b4', alpha=0.85)

        ax2.set_xticks(x_indices)
        ax2.set_xticklabels(all_bitstrings, rotation=45, ha='right', fontsize=9)
        ax2.set_xlabel('Sampled Bitstring (Green labels = Max-Cut)', fontsize=11)
        ax2.set_ylabel('Probability', fontsize=11)
        ax2.set_title('Bitstring Sampling: Ideal vs. Noisy vs. Mitigated', fontsize=12, fontweight='bold')
        ax2.grid(axis='y', alpha=0.3)
        ax2.legend()

        plt.tight_layout()
        display(fig)
        plt.close(fig)

        print("\n" + "=" * 75)
        print("Optimization execution and evaluation completed successfully!")
        print("=" * 75)

button_run.on_click(execute_hybrid_system)

# Display Interactive Dashboard
print("=" * 75)
print("HYBRID QUANTUM OPTIMIZATION INTERACTIVE DASHBOARD")
print("Select problem, algorithm, execution mode, and press 'Run Hybrid Optimization'.")
print("=" * 75)
display(dropdown_problem)
display(dropdown_algo)
display(dropdown_backend)
display(slider_shots)
display(button_run)
display(output_box)

HYBRID QUANTUM OPTIMIZATION INTERACTIVE DASHBOARD
Select problem, algorithm, execution mode, and press 'Run Hybrid Optimization'.


Dropdown(description='Problem Graph:', index=1, layout=Layout(width='450px'), options=('2-Node Toy Graph (1 Ed…

Dropdown(description='Algorithm:', layout=Layout(width='450px'), options=('QAOA (p=1)', 'QAOA (p=2)', 'VQE (Re…

Dropdown(description='Execution Mode:', layout=Layout(width='450px'), options=('Comparative Benchmark (Ideal v…

IntSlider(value=2048, description='Measurement Shots:', layout=Layout(width='450px'), max=4096, min=512, step=…

Button(button_style='primary', description='Run Hybrid Optimization', icon='play', layout=Layout(height='42px'…

Output()